# Streaming — 03: Stateful Operations

## What you will learn
Stateful streaming operations **remember information across micro-batches**.
This enables sophisticated analytics that are impossible with stateless operations.

In this notebook you will master:
1. **`mapGroupsWithState`** — custom state per key, full control over state logic
2. **`flatMapGroupsWithState`** — emit multiple rows per state update
3. **Session windows** — dynamic windows that close after inactivity gaps
4. **State timeout** — how to expire stale state and free memory
5. **State store backends** — HDFS vs RocksDB for large state

## Why stateful streaming?

Stateless streaming (`filter`, `map`, `groupBy` with watermark) works per event.
Stateful streaming lets you answer questions like:

| Question | Requires state |
|---|---|
| How many events in last 5 min? | Windowed aggregation (simple state) |
| Is this user's session still active? | Custom session state |
| What is the user's running total spend? | Accumulating state per user |
| Alert when a user's event rate doubles | Comparative state across time |
| Track a multi-step funnel completion | Sequential state machine |

## The State Model
```
Micro-batch N arrives with events for user_id = 42
        │
        ▼
State Store lookup: "what do I know about user 42?"
        │
        ▼
State update function runs:
  current_state + new_events → new_state + output_rows
        │
        ▼
State Store write: save new_state for user 42
Output: emit output_rows to sink
```
State persists in the **State Store** between batches.
Checkpointing saves the State Store to durable storage for fault tolerance.


In [ ]:
import os, time, datetime, pathlib, shutil, random, json as pyjson
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

GLUTEN_ENABLED = os.environ.get('GLUTEN_ENABLED', 'false').lower() == 'true'
MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data'

spark = (
    SparkSession.builder
    .appName('stateful-streaming')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | Gluten: {GLUTEN_ENABLED}")

## Step 1 — Setup: Streaming Source

We simulate a clickstream where users navigate through a website.
Each event has a user_id, event_type, page, and timestamp.


In [ ]:
# Setup
stream_input = f"{DATA_DIR}/stream_stateful_input"
stream_ckpt  = f"{DATA_DIR}/stream_stateful_checkpoint"

for d in [stream_input, stream_ckpt]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

event_schema = StructType([
    StructField("event_id",   LongType()),
    StructField("user_id",    LongType()),
    StructField("event_type", StringType()),
    StructField("page",       StringType()),
    StructField("revenue",    DoubleType()),
    StructField("event_ts",   TimestampType()),
])

EVENTS = ["page_view","click","add_to_cart","purchase","search"]
PAGES  = ["/home","/products","/cart","/checkout","/account"]

def write_events(batch_id, events):
    path = f"{stream_input}/batch_{batch_id:04d}.json"
    with open(path, "w") as f:
        for e in events:
            f.write(pyjson.dumps(e) + "\n")
    return path

print("Stream source ready.")
print(f"Input dir : {stream_input}")
print(f"Checkpoint: {stream_ckpt}")

## Step 2 — Session Windows

A **session window** groups events that are close together in time.
The window closes (ends) when there is a gap of more than `gap_duration`
with no events for that key.

**Example:** A user browsing a website:
- Event at 10:00 → session starts
- Event at 10:02 → same session (gap < 5 min)
- Event at 10:04 → same session
- Gap > 5 min → session closes
- Event at 10:15 → new session starts

Session windows are more natural than fixed tumbling/sliding windows
for user behaviour analysis because user sessions have variable length.


In [ ]:
stream_input_sess = f"{DATA_DIR}/stream_session_input"
stream_ckpt_sess  = f"{DATA_DIR}/stream_session_checkpoint"
for d in [stream_input_sess, stream_ckpt_sess]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

raw_stream = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .load(stream_input_sess)
)

# Session window: 5-minute gap = session boundary
session_agg = (
    raw_stream
    .withWatermark("event_ts", "10 minutes")   # wait 10 min for late events
    .groupBy(
        "user_id",
        F.session_window("event_ts", "5 minutes")  # 5-min inactivity = new session
    )
    .agg(
        F.count("*").alias("event_count"),
        F.sum(F.when(F.col("revenue") > 0, F.col("revenue")).otherwise(0))
         .alias("session_revenue"),
        F.collect_list("event_type").alias("event_sequence"),
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end")
    )
    .select(
        "user_id",
        F.col("session_window.start").alias("session_start"),
        F.col("session_window.end").alias("session_end"),
        "event_count", "session_revenue", "event_sequence"
    )
)

query_sess = (
    session_agg
    .writeStream
    .format("console")
    .outputMode("complete")  # session_window requires complete mode in Spark 4.0
    .option("truncate", False)
    .option("numRows", 10)
    .option("checkpointLocation", stream_ckpt_sess)
    .queryName("session_windows")
    .start()
)
print(f"Session window query started: {query_sess.name}")

In [ ]:
random.seed(42)
now = datetime.datetime.now()

# Simulate 3 users with distinct session patterns:
# User 1: continuous browsing (one long session)
# User 2: two separate sessions (gap > 5 min)
# User 3: quick purchase (short intense session)

def make_event(eid, uid, etype, page, revenue, ts):
    return {"event_id": eid, "user_id": uid, "event_type": etype,
            "page": page, "revenue": revenue,
            "event_ts": ts.strftime("%Y-%m-%dT%H:%M:%S.%f")}

batch1_events = [
    # User 1: starts browsing
    make_event(1, 1, "page_view", "/home",     0.0, now - datetime.timedelta(minutes=8)),
    make_event(2, 1, "click",     "/products", 0.0, now - datetime.timedelta(minutes=7)),
    make_event(3, 1, "add_to_cart","/cart",    0.0, now - datetime.timedelta(minutes=6)),
    # User 2: first session
    make_event(4, 2, "page_view", "/home",     0.0, now - datetime.timedelta(minutes=20)),
    make_event(5, 2, "search",    "/search",   0.0, now - datetime.timedelta(minutes=19)),
    # User 3: quick purchase
    make_event(6, 3, "page_view", "/products", 0.0, now - datetime.timedelta(minutes=5)),
    make_event(7, 3, "purchase",  "/checkout", 299.99, now - datetime.timedelta(minutes=4)),
]
write_events(1, batch1_events)
print("Batch 1 written: 3 users starting sessions")
time.sleep(4)

batch2_events = [
    # User 1: continues session (< 5 min gap)
    make_event(8,  1, "purchase",  "/checkout", 149.99, now - datetime.timedelta(minutes=4)),
    # User 2: second session (> 5 min gap from first session)
    make_event(9,  2, "page_view", "/home",     0.0, now - datetime.timedelta(minutes=2)),
    make_event(10, 2, "purchase",  "/checkout", 89.99, now - datetime.timedelta(minutes=1)),
]
write_events(2, batch2_events)
print("Batch 2 written: user 1 purchases, user 2 starts new session")
time.sleep(6)

query_sess.stop()
print("\nSession window stream stopped.")

## Step 3 — `mapGroupsWithState`: Custom State Logic

`mapGroupsWithState` gives you **full control** over state management.
You define:
- **State type**: what to store (case class or StructType)
- **Output type**: what to emit
- **Update function**: how to update state given new events
- **Timeout**: when to expire stale state

**Use case:** Track each user's running statistics — total spend, event count,
last seen time. Alert when thresholds are crossed.


In [ ]:
stream_input_map = f"{DATA_DIR}/stream_map_input"
stream_ckpt_map  = f"{DATA_DIR}/stream_map_checkpoint"
for d in [stream_input_map, stream_ckpt_map]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# State schema: what we track per user
user_state_schema = StructType([
    StructField("user_id",       LongType()),
    StructField("event_count",   LongType()),
    StructField("total_revenue", DoubleType()),
    StructField("last_seen",     TimestampType()),
    StructField("is_vip",        BooleanType()),   # becomes VIP at $500 total spend
])

# Output schema: what we emit per batch
user_output_schema = StructType([
    StructField("user_id",       LongType()),
    StructField("event_count",   LongType()),
    StructField("total_revenue", DoubleType()),
    StructField("last_seen",     TimestampType()),
    StructField("is_vip",        BooleanType()),
    StructField("status_change", StringType()),    # "became_vip", "active", "expired"
])

from pyspark.sql.streaming.state import GroupState, GroupStateTimeout

def update_user_state(user_id: int, events, state: GroupState):
    """
    Called once per key (user_id) per micro-batch.
    events: iterator of rows for this user_id in this batch
    state: current persisted state for this user_id
    """
    VIP_THRESHOLD = 500.0
    TIMEOUT_MS    = 10 * 60 * 1000  # expire state after 10 min inactivity

    # Load existing state or initialise
    if state.exists:
        s = state.get
        count   = s["event_count"]
        revenue = s["total_revenue"]
        was_vip = s["is_vip"]
    else:
        count, revenue, was_vip = 0, 0.0, False

    # Handle timeout: user inactive too long
    if state.hasTimedOut:
        state.remove()
        yield (user_id, count, revenue, None, was_vip, "expired")
        return

    # Process new events
    last_seen = None
    for event in events:
        count += 1
        revenue += float(event["revenue"] or 0.0)
        ts = event["event_ts"]
        if last_seen is None or ts > last_seen:
            last_seen = ts

    is_vip = revenue >= VIP_THRESHOLD
    status = "became_vip" if is_vip and not was_vip else "active"

    # Save updated state
    state.update((user_id, count, revenue, last_seen, is_vip))
    state.setTimeoutDuration(TIMEOUT_MS)

    yield (user_id, count, revenue, last_seen, is_vip, status)

raw_stream_map = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .load(stream_input_map)
)

user_stats = (
    raw_stream_map
    .withWatermark("event_ts", "5 minutes")
    .groupBy("user_id")
    .applyInPandasWithState(
        update_user_state,
        user_output_schema,
        user_state_schema,
        "update",
        GroupStateTimeout.ProcessingTimeTimeout
    )
)

query_map = (
    user_stats
    .writeStream
    .format("console")
    .outputMode("update")
    .option("truncate", False)
    .option("checkpointLocation", stream_ckpt_map)
    .queryName("user_stats")
    .start()
)
print(f"mapGroupsWithState query started: {query_map.name}")

In [ ]:
random.seed(99)
now = datetime.datetime.now()

# Generate events where some users cross the VIP threshold ($500 total)
def make_ev(eid, uid, rev, secs_ago=0):
    ts = now - datetime.timedelta(seconds=secs_ago)
    return {"event_id": eid, "user_id": uid, "event_type": "purchase",
            "page": "/checkout", "revenue": rev,
            "event_ts": ts.strftime("%Y-%m-%dT%H:%M:%S.%f")}

print("Batch 1: initial purchases")
write_events(1, [
    make_ev(1,  101, 200.0, 60),   # user 101: $200
    make_ev(2,  102,  50.0, 55),   # user 102: $50
    make_ev(3,  103, 150.0, 50),   # user 103: $150
])
time.sleep(4)

print("Batch 2: more purchases — user 101 crosses VIP threshold!")
write_events(2, [
    make_ev(4, 101, 350.0, 30),    # user 101: $200+$350=$550 → VIP!
    make_ev(5, 102, 100.0, 25),    # user 102: $150
    make_ev(6, 103, 200.0, 20),    # user 103: $350
    make_ev(7, 104,  75.0, 15),    # user 104: new user $75
])
time.sleep(4)

print("Batch 3: user 103 also crosses VIP threshold")
write_events(3, [
    make_ev(8,  103, 200.0, 10),   # user 103: $350+$200=$550 → VIP!
    make_ev(9,  104, 500.0, 5),    # user 104: $575 → VIP!
])
time.sleep(6)

query_map.stop()
print("\nmapGroupsWithState stream stopped.")
print("Check output above: look for status_change='became_vip' rows.")

## Step 4 — `flatMapGroupsWithState`: Funnel Analysis

`flatMapGroupsWithState` is like `mapGroupsWithState` but can emit
**zero or many rows** per state update.

**Use case:** Detect when users complete a conversion funnel:
`page_view → add_to_cart → purchase`

We track each user's progress through the funnel. When they complete it,
we emit a conversion event. If they abandon (timeout), we emit an abandonment event.


In [ ]:
stream_input_flat = f"{DATA_DIR}/stream_flat_input"
stream_ckpt_flat  = f"{DATA_DIR}/stream_flat_checkpoint"
for d in [stream_input_flat, stream_ckpt_flat]:
    shutil.rmtree(d, ignore_errors=True)
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)

# Funnel stages in order
FUNNEL_STAGES = ["page_view", "add_to_cart", "purchase"]

funnel_state_schema = StructType([
    StructField("user_id",        LongType()),
    StructField("stages_seen",    StringType()),    # comma-separated stages
    StructField("funnel_start",   TimestampType()),
    StructField("last_event",     TimestampType()),
])

funnel_output_schema = StructType([
    StructField("user_id",        LongType()),
    StructField("outcome",        StringType()),    # "converted" or "abandoned"
    StructField("stages_seen",    StringType()),
    StructField("duration_secs",  LongType()),
    StructField("detected_at",    TimestampType()),
])

def track_funnel(user_id, events, state: GroupState):
    """
    Track funnel progress. Emit when funnel is complete or abandoned.
    flatMapGroupsWithState: can yield 0 or many rows per call.
    """
    TIMEOUT_MS = 15 * 60 * 1000  # 15 min to complete funnel

    if state.hasTimedOut:
        if state.exists:
            s = state.get
            stages = s["stages_seen"].split(",") if s["stages_seen"] else []
            duration = int((datetime.datetime.now() - s["funnel_start"]).total_seconds())
            yield (user_id, "abandoned", s["stages_seen"], duration, datetime.datetime.now())
        state.remove()
        return

    # Load or init state
    if state.exists:
        s = state.get
        stages_seen  = set(s["stages_seen"].split(",")) if s["stages_seen"] else set()
        funnel_start = s["funnel_start"]
    else:
        stages_seen  = set()
        funnel_start = None

    for event in events:
        etype = event["event_type"]
        ts    = event["event_ts"]
        if etype in FUNNEL_STAGES:
            stages_seen.add(etype)
            if funnel_start is None:
                funnel_start = ts

    stages_str = ",".join(sorted(stages_seen))
    all_complete = all(s in stages_seen for s in FUNNEL_STAGES)

    if all_complete:
        duration = int((datetime.datetime.now() - funnel_start).total_seconds()) if funnel_start else 0
        yield (user_id, "converted", stages_str, duration, datetime.datetime.now())
        state.remove()   # clear state after conversion
    else:
        # Update state and set timeout
        state.update((user_id, stages_str, funnel_start or datetime.datetime.now(), datetime.datetime.now()))
        state.setTimeoutDuration(TIMEOUT_MS)
        # No output yet — waiting for funnel completion

raw_stream_flat = (
    spark.readStream
         .format("json")
         .schema(event_schema)
         .option("maxFilesPerTrigger", 1)
         .load(stream_input_flat)
)

funnel_stream = (
    raw_stream_flat
    .withWatermark("event_ts", "5 minutes")
    .groupBy("user_id")
    .applyInPandasWithState(
        track_funnel,
        funnel_output_schema,
        funnel_state_schema,
        "append",
        GroupStateTimeout.ProcessingTimeTimeout
    )
)

query_flat = (
    funnel_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .option("checkpointLocation", stream_ckpt_flat)
    .queryName("funnel_tracker")
    .start()
)
print(f"flatMapGroupsWithState funnel tracker started: {query_flat.name}")

In [ ]:
now = datetime.datetime.now()

def funnel_event(eid, uid, etype, secs_ago=0):
    ts = now - datetime.timedelta(seconds=secs_ago)
    return {"event_id": eid, "user_id": uid, "event_type": etype,
            "page": "/page", "revenue": 99.99 if etype=="purchase" else 0.0,
            "event_ts": ts.strftime("%Y-%m-%dT%H:%M:%S.%f")}

print("Batch 1: users start the funnel")
write_events(1, [
    funnel_event(1, 201, "page_view",   60),   # user 201: step 1
    funnel_event(2, 202, "page_view",   55),   # user 202: step 1
    funnel_event(3, 203, "page_view",   50),   # user 203: step 1
    funnel_event(4, 201, "add_to_cart", 45),   # user 201: step 2
])
time.sleep(4)

print("Batch 2: some users progress, some complete")
write_events(2, [
    funnel_event(5, 201, "purchase",   30),    # user 201 CONVERTS! (all 3 steps)
    funnel_event(6, 202, "add_to_cart",25),    # user 202: step 2
    funnel_event(7, 203, "add_to_cart",20),    # user 203: step 2
])
time.sleep(4)

print("Batch 3: user 202 converts, user 203 abandons (only 2 steps)")
write_events(3, [
    funnel_event(8, 202, "purchase", 10),      # user 202 CONVERTS!
    # user 203 never purchases → will timeout and emit "abandoned"
])
time.sleep(6)

query_flat.stop()
print("\nFunnel tracker stopped.")
print("Check output above: user 201 and 202 should show 'converted'.")
print("User 203 will show 'abandoned' when the timeout fires.")

## Step 5 — State Store Backends

By default Spark uses an **HDFS-based State Store** which stores state
in serialized Avro files on the checkpoint storage.

For large state (millions of keys), **RocksDB State Store** is much better:
- RocksDB stores state on-disk on executor local storage (not in JVM heap)
- Handles state 10-100x larger than HDFS store without OOM
- Faster point lookups (O(log n) vs O(n) for HDFS store)

**When to switch to RocksDB:**
- State size > 1 GB per executor
- Many distinct keys (> 1M)
- Frequent state lookups with low latency requirement


In [ ]:
# Show current state store configuration
print("State Store configuration:")
state_configs = [
    "spark.sql.streaming.stateStore.providerClass",
    "spark.sql.streaming.stateStore.rocksdb.changelogCheckpointing.enabled",
]
for c in state_configs:
    try:
        print(f"  {c.split('.')[-1]:<45} {spark.conf.get(c)}")
    except Exception:
        print(f"  {c.split('.')[-1]:<45} (default)")

print()
print("Default: HDFSBackedStateStoreProvider")
print("For production with large state, switch to RocksDB:")
print()
print("  spark.conf.set(")
print("      'spark.sql.streaming.stateStore.providerClass',")
print("      'org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider'")
print("  )")
print()
print("RocksDB advantages:")
print("  - Stores state on executor local disk (not JVM heap)")
print("  - Handles 10-100x more state per executor")
print("  - Better for high-cardinality keys (millions of users)")
print()
print("Note: RocksDB requires additional native libraries.")
print("In this cluster it may not be available — use HDFS store for learning.")

## Summary: Stateful Streaming Patterns

### When to use each approach

| Pattern | API | Best for |
|---|---|---|
| Fixed tumbling/sliding window | `groupBy` + `window()` | Regular time-based aggregations |
| Session window | `groupBy` + `session_window()` | User sessions, activity grouping |
| Custom state per key | `mapGroupsWithState` / `applyInPandasWithState` | Complex business logic, alerts |
| Multiple outputs per update | `flatMapGroupsWithState` | Funnel detection, state machines |

### State timeout strategies

| Timeout type | When it fires | Best for |
|---|---|---|
| `ProcessingTimeTimeout` | After N ms of processing time | Expiring inactive sessions |
| `EventTimeTimeout` | When watermark passes key's timestamp | Event-time based expiry |

### Production checklist

1. **Always set a timeout** — unbounded state growth will OOM your executors
2. **Use RocksDB** for > 1M keys or > 1 GB state per executor
3. **Monitor state size** — `query.lastProgress['stateOperators']` shows state row count
4. **Test timeout handling** — it runs in a separate call with `state.hasTimedOut=True`
5. **Checkpoint regularly** — state is only durable when checkpointed

### Monitoring state size
```python
prog = query.lastProgress
if prog and 'stateOperators' in prog:
    for op in prog['stateOperators']:
        print(f"  numRowsTotal   : {op['numRowsTotal']}")
        print(f"  memoryUsedBytes: {op['memoryUsedBytes']}")
```
